In [3]:
import sys
import statistics
import time
import re
from collections import Counter, defaultdict

import HimalayanTokenization
from transformers import AutoTokenizer
import json

# Configuration
VOCAB_TSV = '/home/lang-chain/Documents/HimalayanTokenizer/trained_vocab.tsv'
DATASET = 'sample.txt'
FOLDING_RULES = [
    ("सङ्ग", "संग"),
    ("सँग", "संग"),
]

# Initialize tokenizers
print("Loading tokenizers...")
tok = HimalayanTokenization.PyHimalayanTokenization(mode="LM")
tok.load_vocab_tsv(VOCAB_TSV)
indic_tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")


Loading tokenizers...


[transformers] Could not extract SentencePiece model from /home/lang-chain/.cache/huggingface/hub/models--ai4bharat--indic-bert/snapshots/4842dd258ecc0546f0d660b76a3b22a9c632f401/spiece.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: Error parsing line b'\x0e' in /home/lang-chain/.cache/huggingface/hub/models--ai4bharat--indic-bert/snapshots/4842dd258ecc0546f0d660b76a3b22a9c632f401/spiece.model

In [2]:
with open(DATASET, "r", encoding="utf-8") as f:
    corpus = [line.strip() for line in f if line.strip()]

# Combine into one text for some metrics
full_text = " ".join(corpus)
words = full_text.split()

In [3]:
print(f"\nCorpus Stats:")
print(f"Total lines: {len(corpus)}")
print(f"Total words: {len(words)}")
print(f"Total characters: {len(full_text)}")
print("="*60)


Corpus Stats:
Total lines: 596
Total words: 5894
Total characters: 40407


In [5]:
print("\n📊 1. VOCABULARY STATS")
print("-"*40)
indic_vocab_size = indic_tokenizer.vocab_size
print(f"NepBPE Vocab Size: {tok.vocab_size()}")
print(f"IndicBERT Vocab Size: {indic_vocab_size}")


📊 1. VOCABULARY STATS
----------------------------------------
NepBPE Vocab Size: 64009
IndicBERT Vocab Size: 200000


In [6]:
print("\n⏱️  2. TOKENIZATION SPEED")
print("-"*40)

# Warm up
for _ in range(100):
    _ = tok.encode("नमस्ते नेपाल")

start = time.time()
for line in corpus:
    _ = tok.encode(line)
nepbpe_time = time.time() - start

start = time.time()
for line in corpus:
    _ = indic_tokenizer.encode(line, add_special_tokens=False)
indic_time = time.time() - start

print(f"NepBPE: {nepbpe_time:.4f}s total, {len(corpus)/nepbpe_time:.0f} lines/s")
print(f"IndicBERT: {indic_time:.4f}s total, {len(corpus)/indic_time:.0f} lines/s")
print(f"Speed ratio: {indic_time/nepbpe_time:.2f}x (NepBPE is {'faster' if nepbpe_time < indic_time else 'slower'})")


⏱️  2. TOKENIZATION SPEED
----------------------------------------
NepBPE: 0.0057s total, 104224 lines/s
IndicBERT: 0.0414s total, 14409 lines/s
Speed ratio: 7.23x (NepBPE is faster)


In [24]:
print("\n🔍 3. DETAILED TOKENIZATION STATS")
print("-"*40)

nepbpe_all_tokens = []
indic_all_tokens = []
nepbpe_unk_count = 0
indic_unk_count = 0

# Define your UNK token (if you have one)
# NepBPE uses byte-fallback, so there is NO UNK token.
# We set it to -1 to indicate "not applicable"
NEPBPE_UNK_ID = -1

# IndicBERT UNK token (usually 3, but check your tokenizer)
INDIC_UNK_ID = indic_tokenizer.unk_token_id if hasattr(indic_tokenizer, 'unk_token_id') else 3

for line in corpus:
    # NepBPE
    try:
        nepbpe_tokens = tok.encode(line)
        nepbpe_all_tokens.extend(nepbpe_tokens)
        # NepBPE has NO UNK token because of byte-fallback
        # So we don't count any as "unknown"
        nepbpe_unk_count += 0
    except:
        pass
    
    # IndicBERT
    try:
        indic_tokens = indic_tokenizer.encode(line, add_special_tokens=False)
        indic_all_tokens.extend(indic_tokens)
        indic_unk_count += sum(1 for t in indic_tokens if t == INDIC_UNK_ID)
    except:
        pass

# Total token counts
nepbpe_total = len(nepbpe_all_tokens)
indic_total = len(indic_all_tokens)

# UNK rates
nepbpe_unk_rate = 0.0  # Always 0 because byte-fallback
indic_unk_rate = (indic_unk_count / indic_total * 100) if indic_total > 0 else 0

print(f"Total tokens produced:")
print(f"  NepBPE: {nepbpe_total}")
print(f"  IndicBERT: {indic_total}")

print(f"\nUNK token count:")
print(f"  NepBPE: {nepbpe_unk_count} (byte-fallback means NO UNK tokens)")
print(f"  IndicBERT: {indic_unk_count}")

print(f"\nUNK token rate:")
print(f"  NepBPE: {nepbpe_unk_rate:.2f}%  <- 0% because every byte maps to a valid token")
print(f"  IndicBERT: {indic_unk_rate:.2f}%")


🔍 3. DETAILED TOKENIZATION STATS
----------------------------------------
Total tokens produced:
  NepBPE: 10242
  IndicBERT: 10766

UNK token count:
  NepBPE: 0 (byte-fallback means NO UNK tokens)
  IndicBERT: 102

UNK token rate:
  NepBPE: 0.00%  <- 0% because every byte maps to a valid token
  IndicBERT: 0.95%


In [25]:
print("\n📦 4. COMPRESSION EFFICIENCY")
print("-"*40)
total_chars = len(full_text)
nepbpe_compression = total_chars / len(nepbpe_all_tokens) if nepbpe_all_tokens else 0
indic_compression = total_chars / len(indic_all_tokens) if indic_all_tokens else 0

print(f"Compression ratio (chars/token):")
print(f"  NepBPE: {nepbpe_compression:.2f}")
print(f"  IndicBERT: {indic_compression:.2f}")


📦 4. COMPRESSION EFFICIENCY
----------------------------------------
Compression ratio (chars/token):
  NepBPE: 3.95
  IndicBERT: 3.75


In [19]:
print("\n📈 5. TOKEN FREQUENCY DISTRIBUTION")
print("-"*40)

# Get token frequencies from NepBPE
nepbpe_freq = Counter(nepbpe_all_tokens)

print("Top 10 most frequent tokens:")
print("NepBPE:")
for token_id, freq in nepbpe_freq.most_common(10):
    surface = tok.get_token_surface(token_id)
    print(f"  {surface} (ID:{token_id}): {freq}")

print("IndicBERT:")
try:
    indic_freq = Counter(indic_all_tokens)
    for token_id, freq in indic_freq.most_common(10):
        decoded = indic_tokenizer.decode([token_id])
        print(f"  {decoded} (ID:{token_id}): {freq}")
except Exception as e:
    print(f"  [IndicBERT analysis unavailable: {e}]")


📈 5. TOKEN FREQUENCY DISTRIBUTION
----------------------------------------
Top 10 most frequent tokens:
NepBPE:
  ▂ (ID:48000): 622
  . (ID:478): 371
  ▁ (ID:477): 319
  s (ID:540): 225
  , (ID:479): 155
  ▂the (ID:48032): 130
  ▂The (ID:48070): 97
  ã (ID:720): 97
  Ģ (ID:621): 97
  Ĥ (ID:623): 97
IndicBERT:
  . (ID:9): 375
  the (ID:11): 255
   (ID:8): 194
  क (ID:507): 162
  य (ID:1134): 160
  s (ID:32): 144
  । (ID:15): 143
  , (ID:12): 139
  न (ID:339): 109
  गर (ID:39324): 103


In [18]:
print("\n🔤 6. SUBWORD ANALYSIS")
print("-"*40)

# For IndicBERT: check how many tokens start with ## (continued subwords)
# Using the same sample texts directly
sample_texts = [
    "नेपाल एक सुन्दर देश हो",
    "म घर जाँदै छु",
    "हामीले विद्यालयमा पढ्यौं",
    "संविधान सभाबाट पारित भयो",
]

# Calculate for IndicBERT
indic_continued = 0
indic_total_tokens_sample = 0

try:
    for text in sample_texts:
        tokens = indic_tokenizer.tokenize(text)
        indic_continued += sum(1 for t in tokens if t.startswith('##'))
        indic_total_tokens_sample += len(tokens)
    
    indic_continued_rate = (indic_continued / indic_total_tokens_sample * 100) if indic_total_tokens_sample else 0
    print(f"IndicBERT continued subword rate: {indic_continued_rate:.2f}%")
    print(f"  (Tokens starting with '##' = {indic_continued} / {indic_total_tokens_sample})")
except Exception as e:
    print(f"IndicBERT: {e}")
# Count ▁/▂ markers in the sample texts
nepbpe_total_tokens = 0
nepbpe_continued = 0
nepbpe_new_words = 0

for text in sample_texts:
    ids = tok.encode(text)
    surfaces = [tok.get_token_surface(i) for i in ids]
    nepbpe_total_tokens += len(ids)
    
    # In NepBPE, tokens WITHOUT ▁/▂ are continuations
    # Tokens WITH ▁/▂ are new word starts
    for s in surfaces:
        if s.startswith('▁') or s.startswith('▂'):
            nepbpe_new_words += 1
        else:
            nepbpe_continued += 1

nepbpe_continued_rate = (nepbpe_continued / nepbpe_total_tokens * 100) if nepbpe_total_tokens else 0

print(f"\nNepBPE continued subword rate: {nepbpe_continued_rate:.2f}%")
print(f"  Tokens WITHOUT ▁/▂ (continuations): {nepbpe_continued} / {nepbpe_total_tokens} total tokens")
print(f"  Tokens WITH ▁/▂ (new word starts): {nepbpe_new_words} / {nepbpe_total_tokens}")



🔤 6. SUBWORD ANALYSIS
----------------------------------------
IndicBERT continued subword rate: 0.00%
  (Tokens starting with '##' = 0 / 25)

NepBPE continued subword rate: 5.88%
  Tokens WITHOUT ▁/▂ (continuations): 1 / 17 total tokens
  Tokens WITH ▁/▂ (new word starts): 16 / 17


In [14]:
print("\n📝 7. SAMPLE TOKENIZATION COMPARISONS")
print("-"*40)

sample_texts = [
    "नेपाल एक सुन्दर देश हो",
    "म घर जाँदै छु",
    "हामीले विद्यालयमा पढ्यौं",
    "संविधान सभाबाट पारित भयो",
]

# Helper to show NepBPE surfaces
def get_nepbpe_surfaces(tok, text):
    ids = tok.encode(text)
    surfaces = [tok.get_token_surface(i) for i in ids]
    return surfaces

for text in sample_texts:
    print(f"\nInput: {text}")
    
    # NepBPE with surfaces (like IndicBERT)
    try:
        nepbpe_ids = tok.encode(text)
        nepbpe_surfaces = [tok.get_token_surface(i) for i in nepbpe_ids]
        print(f"  NepBPE IDs: {nepbpe_ids}")
        print(f"  NepBPE Surfaces: {nepbpe_surfaces}")
    except Exception as e:
        print(f"  NepBPE Error: {e}")
    
    # IndicBERT (original format from your log)
    try:
        # If you have indic_tokenizer loaded, use it.
        # Otherwise, comment this out or use a placeholder.
        indic_tokens = indic_tokenizer.tokenize(text)
        print(f"  IndicBERT: {indic_tokens}")
    except Exception as e:
        print(f"  IndicBERT: [not loaded]")


# ============================================
# 8. VOCABULARY OVERLAP (if both use subwords)
# ============================================
print("\n🔄 8. VOCABULARY ANALYSIS")
print("-"*40)

# Get character coverage from NepBPE vocab
nepbpe_chars = set()
if hasattr(tok, 'vocab') or hasattr(tok, 'stoi'):
    # Try to get vocab from the Rust tokenizer
    try:
        for i in range(tok.vocab_size()):
            surface = tok.get_token_surface(i)
            if surface:
                nepbpe_chars.update(surface)
    except:
        # If tok doesn't support iteration, just print what we can
        print("  Cannot iterate vocab; surfaces are shown above.")
else:
    print("  No vocab accessible.")

print(f"NepBPE unique characters in vocab: {len(nepbpe_chars)}")
if nepbpe_chars:
    print(f"Sample chars: {sorted(list(nepbpe_chars))[:20]}")


📝 7. SAMPLE TOKENIZATION COMPARISONS
----------------------------------------

Input: नेपाल एक सुन्दर देश हो
  NepBPE IDs: [836, 898, 4019, 1173, 854]
  NepBPE Surfaces: ['▁नेपाल', '▁एक', '▁सुन्दर', '▁देश', '▁हो']
  IndicBERT: ['▁न', 'पल', '▁एक', '▁सन', 'दर', '▁दश', '▁ह']

Input: म घर जाँदै छु
  NepBPE IDs: [774, 1270, 5986, 2121]
  NepBPE Surfaces: ['▁म', '▁घर', '▁जाँदै', '▁छु']
  IndicBERT: ['▁म', '▁घर', '▁जद', '▁छ']

Input: हामीले विद्यालयमा पढ्यौं
  NepBPE IDs: [1988, 4323, 16674, 3816]
  NepBPE Surfaces: ['▁हामीले', '▁विद्यालयमा', '▁पढ्', 'यौं']
  IndicBERT: ['▁हम', 'ल', '▁व', 'द', 'यल', 'यम', '▁पढ', 'य']

Input: संविधान सभाबाट पारित भयो
  NepBPE IDs: [1607, 12860, 3462, 1458]
  NepBPE Surfaces: ['▁संविधान', '▁सभाबाट', '▁पारित', '▁भयो']
  IndicBERT: ['▁सव', 'धन', '▁सभ', 'बट', '▁परत', '▁भय']

🔄 8. VOCABULARY ANALYSIS
----------------------------------------
  No vocab accessible.
NepBPE unique characters in vocab: 0


In [13]:
print("\n💾 9. MEMORY/STORAGE")
print("-"*40)
import os

# Vocabulary file size
if os.path.exists(VOCAB_TSV):
    nepbpe_vocab_file_size = os.path.getsize(VOCAB_TSV)
    print(f"NepBPE vocabulary file size: {nepbpe_vocab_file_size/1024:.2f} KB")
    
# IndicBERT vocab size (cached)
indic_vocab_path = indic_tokenizer.vocab_files_names.get('vocab_file', 'N/A')
print(f"IndicBERT vocab file: {indic_vocab_path}")


💾 9. MEMORY/STORAGE
----------------------------------------
NepBPE vocabulary file size: 1421.66 KB
IndicBERT vocab file: spiece.model


In [28]:

summary = {
    'vocab_size': {
        'nepbpe': tok.vocab_size(),
        'indicbert': indic_vocab_size
    },
    'total_tokens': {
        'nepbpe': len(nepbpe_all_tokens),
        'indicbert': len(indic_all_tokens)
    },
    'avg_tokens_per_word': {
        'nepbpe': round(nepbpe_avg_tokens, 3),
        'indicbert': round(indic_avg_tokens, 3)
    },
    'compression_ratio': {
        'nepbpe': round(nepbpe_compression, 2),
        'indicbert': round(indic_compression, 2)
    },
    'unk_rate': {
        'nepbpe': f"{nepbpe_unk_rate:.2f}%",
        'indicbert': f"{indic_unk_rate:.2f}%"
    },
    'tokenization_speed': {
        'nepbpe': f"{len(corpus)/nepbpe_time:.0f} lines/s",
        'indicbert': f"{len(corpus)/indic_time:.0f} lines/s"
    },
    'speedup': f"{indic_time/nepbpe_time:.2f}x" if nepbpe_time < indic_time else f"{nepbpe_time/indic_time:.2f}x"
}

In [29]:
summary

{'vocab_size': {'nepbpe': 64009, 'indicbert': 200000},
 'total_tokens': {'nepbpe': 10242, 'indicbert': 10766},
 'avg_tokens_per_word': {'nepbpe': 1.738, 'indicbert': 1.827},
 'compression_ratio': {'nepbpe': 3.95, 'indicbert': 3.75},
 'unk_rate': {'nepbpe': '0.00%', 'indicbert': '0.95%'},
 'tokenization_speed': {'nepbpe': '104224 lines/s',
  'indicbert': '14409 lines/s'},
 'speedup': '7.23x'}

In [ ]:
import sys
import statistics
import time
import re
import os
from collections import Counter, defaultdict

import HimalayanTokenization
from transformers import AutoTokenizer
import json

# Configuration
VOCAB_TSV = 'vocab_nepbpe/nepbpe_vocab_bilingual_v10.tsv'

from transformers import AutoTokenizer
import json


DATASET = 'sample.txt'
FOLDING_RULES = [
    ("सङ्ग", "संग"),
    ("सँग", "संग"),
]

# ============================================
# LOAD ALL TOKENIZERS
# ============================================
print("="*60)
print("LOADING TOKENIZERS")
print("="*60)

# 1. HimalayanTOK – load and determine actual vocab size
print("Loading HimalayanTOK-Nepali...")
tok =  HimalayanTokenization.PyHimalayanTokenization(mode="LM")
tok.load_vocab_tsv(VOCAB_TSV)
ht_vocab_size = tok.vocab_size()
if ht_vocab_size != 64000:
    print(f"  ⚠️  WARNING: Loaded vocabulary size is {ht_vocab_size:,}, not 64,000.")
    print(f"     The tokenizer will be named 'HimalayanTOK-{ht_vocab_size//1000}K' to reflect reality.")
HT_NAME = f"HimalayanTOK-{ht_vocab_size//1000}K"

# 2. IndicBERT
print("Loading IndicBERT...")
indic_tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")

# 3. mBERT (Multilingual BERT)
print("Loading mBERT...")
mbert_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

# 4. XLM-RoBERTa
print("Loading XLM-R...")
xlmr_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# 5. Tiktoken (try both bases)
print("Loading Tiktoken...")
try:
    import tiktoken
    tiktoken_cl100k = tiktoken.get_encoding("cl100k_base")
    tiktoken_o200k = tiktoken.get_encoding("o200k_base")
    tiktoken_available = True
except ImportError:
    print("  ⚠️ Tiktoken not installed. Install with: pip install tiktoken")
    tiktoken_available = False
    tiktoken_cl100k = None
    tiktoken_o200k = None

# 6. Llama tokenizer (optional)
print("Loading Llama tokenizer...")
try:
    llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
    llama_available = True
except:
    try:
        # Fallback to a smaller accessible model
        llama_tokenizer = AutoTokenizer.from_pretrained("huggyllama/llama-7b")
        llama_available = True
    except:
        print("  ⚠️ Llama tokenizer not available (requires auth or local files)")
        llama_available = False
        llama_tokenizer = None

# ============================================
# DEFINE TOKENIZERS DICT FOR EASY ITERATION
# ============================================
tokenizers = {
    HT_NAME: {
        "tokenizer": tok,
        "type": "custom",
        "description": f"HimalayanTOK-Nepali (BPE, byte-fallback, 100% Devanagari)",
        "encode": lambda t: tok.encode(t),
        "decode_single": lambda tid: tok.get_token_surface(tid),
        "vocab_size": lambda: tok.vocab_size(),
        "unk_id": -1,  # No UNK (byte-fallback)
    },
    "IndicBERT": {
        "tokenizer": indic_tokenizer,
        "type": "hf",
        "description": "AI4Bharat IndicBERT (200K vocab, WordPiece)",
        "encode": lambda t: indic_tokenizer.encode(t, add_special_tokens=False),
        "decode_single": lambda tid: indic_tokenizer.decode([tid]),
        "vocab_size": lambda: indic_tokenizer.vocab_size,
        "unk_id": indic_tokenizer.unk_token_id,
    },
    "mBERT": {
        "tokenizer": mbert_tokenizer,
        "type": "hf",
        "description": "BERT-base-multilingual-cased (119K vocab, WordPiece)",
        "encode": lambda t: mbert_tokenizer.encode(t, add_special_tokens=False),
        "decode_single": lambda tid: mbert_tokenizer.decode([tid]),
        "vocab_size": lambda: mbert_tokenizer.vocab_size,
        "unk_id": mbert_tokenizer.unk_token_id,
    },
    "XLM-R": {
        "tokenizer": xlmr_tokenizer,
        "type": "hf",
        "description": "XLM-RoBERTa-base (250K vocab, SentencePiece)",
        "encode": lambda t: xlmr_tokenizer.encode(t, add_special_tokens=False),
        "decode_single": lambda tid: xlmr_tokenizer.decode([tid]),
        "vocab_size": lambda: xlmr_tokenizer.vocab_size,
        "unk_id": xlmr_tokenizer.unk_token_id,
    },
}

if tiktoken_available:
    tokenizers["TikToken-cl100k"] = {
        "tokenizer": tiktoken_cl100k,
        "type": "tiktoken",
        "description": "OpenAI TikToken cl100k_base (100K vocab, BPE)",
        "encode": lambda t: tiktoken_cl100k.encode(t),
        "decode_single": lambda tid: tiktoken_cl100k.decode([tid]),
        "vocab_size": lambda: tiktoken_cl100k.n_vocab,
        "unk_id": None,
    }
    tokenizers["TikToken-o200k"] = {
        "tokenizer": tiktoken_o200k,
        "type": "tiktoken",
        "description": "OpenAI TikToken o200k_base (200K vocab, BPE)",
        "encode": lambda t: tiktoken_o200k.encode(t),
        "decode_single": lambda tid: tiktoken_o200k.decode([tid]),
        "vocab_size": lambda: tiktoken_o200k.n_vocab,
        "unk_id": None,
    }

if llama_available:
    tokenizers["Llama-2"] = {
        "tokenizer": llama_tokenizer,
        "type": "hf",
        "description": "Meta Llama-2 (32K vocab, BPE)",
        "encode": lambda t: llama_tokenizer.encode(t, add_special_tokens=False),
        "decode_single": lambda tid: llama_tokenizer.decode([tid]),
        "vocab_size": lambda: llama_tokenizer.vocab_size,
        "unk_id": llama_tokenizer.unk_token_id,
    }

# ============================================
# LOAD CORPUS
# ============================================
print("\nLoading corpus...")
with open(DATASET, "r", encoding="utf-8") as f:
    corpus = [line.strip() for line in f if line.strip()]

full_text = " ".join(corpus)
words = full_text.split()

print(f"\n{'='*60}")
print(f"BENCHMARK: {HT_NAME} vs Major Tokenizers")
print(f"{'='*60}")
print(f"Corpus: {DATASET}")
print(f"Total lines: {len(corpus)}")
print(f"Total words: {len(words)}")
print(f"Total characters: {len(full_text)}")

# ============================================
# 1. VOCABULARY SIZE & TOKENIZER INFO
# ============================================
print("\n" + "="*60)
print("1. TOKENIZER INFORMATION & VOCABULARY SIZE")
print("="*60)

print(f"\n{'Tokenizer':<25} {'Vocab Size':>12} {'Type':<20}")
print(f"{'-'*25} {'-'*12} {'-'*20}")
for name, tok_info in tokenizers.items():
    try:
        vocab_sz = tok_info["vocab_size"]()
        tok_type = tok_info["type"].upper()
        print(f"  {name:<23} {vocab_sz:>12,}  {tok_type:<20}")
        if "description" in tok_info:
            print(f"    └─ {tok_info['description']}")
    except Exception as e:
        print(f"  {name:<23} {'ERROR':>12}")

# ============================================
# 2. TOKENIZATION SPEED
# ============================================
print("\n" + "="*60)
print("2. TOKENIZATION SPEED BENCHMARK (CPU)")
print("="*60)

speed_results = {}
for name, tok_info in tokenizers.items():
    try:
        encode_fn = tok_info["encode"]
        
        # Warm up
        for _ in range(50):
            _ = encode_fn(corpus[0])
        
        # Benchmark
        start = time.time()
        for line in corpus:
            _ = encode_fn(line)
        elapsed = time.time() - start
        
        lines_per_sec = len(corpus) / elapsed if elapsed > 0 else 0
        speed_results[name] = {
            'time': elapsed,
            'lines_per_sec': lines_per_sec
        }
        print(f"  {name:<23} {elapsed:>8.3f}s  ({lines_per_sec:>10,.0f} lines/s)")
    except Exception as e:
        print(f"  {name:<23} ERROR - {e}")

# Find fastest and calculate relative speeds
if speed_results:
    fastest = max(speed_results.items(), key=lambda x: x[1]['lines_per_sec'])
    slowest = min(speed_results.items(), key=lambda x: x[1]['lines_per_sec'])
    
    print(f"\n  {'='*40}")
    print(f"  🏆 FASTEST: {fastest[0]} at {fastest[1]['lines_per_sec']:,.0f} lines/s")
    print(f"  🐌 SLOWEST: {slowest[0]} at {slowest[1]['lines_per_sec']:,.0f} lines/s")
    print(f"  {'='*40}")
    print(f"  ⚠️  Note: Speed depends heavily on implementation (C++ vs Python/HF) and is not a purely algorithmic comparison.")
    
    print(f"\n  Relative Performance (1.00x = fastest):")
    for name, result in sorted(speed_results.items(), key=lambda x: x[1]['lines_per_sec'], reverse=True):
        ratio = fastest[1]['lines_per_sec'] / result['lines_per_sec']
        bar = '█' * int(ratio * 5) if ratio <= 20 else '█' * 100
        print(f"    {name:<23} {ratio:>5.2f}x  {bar}")

# ============================================
# 3. DETAILED TOKENIZATION STATS
# ============================================
print("\n" + "="*60)
print("3. TOKENIZATION EFFICIENCY METRICS")
print("="*60)

stats_results = {}
for name, tok_info in tokenizers.items():
    try:
        encode_fn = tok_info["encode"]
        unk_id = tok_info["unk_id"]
        
        all_tokens = []
        unk_count = 0
        
        for line in corpus:
            tokens = encode_fn(line)
            all_tokens.extend(tokens)
            
            # Count UNKs
            if unk_id is not None and unk_id >= 0:
                unk_count += sum(1 for t in tokens if t == unk_id)
        
        total_tokens = len(all_tokens)
        unk_rate = (unk_count / total_tokens * 100) if total_tokens > 0 else 0
        avg_tokens_per_word = total_tokens / len(words) if words else 0
        compression_ratio = len(full_text) / total_tokens if total_tokens > 0 else 0
        total_bytes = len(full_text.encode('utf-8'))
        bytes_per_token = total_bytes / total_tokens if total_tokens > 0 else 0
        
        stats_results[name] = {
            'total_tokens': total_tokens,
            'unk_count': unk_count,
            'unk_rate': unk_rate,
            'avg_tokens_per_word': avg_tokens_per_word,
            'compression_ratio': compression_ratio,
            'bytes_per_token': bytes_per_token,
            'all_tokens': all_tokens,
        }
        
        unk_str = f"{unk_rate:.2f}%" if unk_id is not None and unk_id >= 0 else "0.00% (byte-fallback)"
        print(f"\n  {name}:")
        print(f"    ├─ Total tokens:        {total_tokens:>10,}")
        print(f"    ├─ Tokens per word:     {avg_tokens_per_word:>10.3f}")
        print(f"    ├─ Compression ratio:   {compression_ratio:>10.2f} chars/token")
        print(f"    ├─ Bytes per token:     {bytes_per_token:>10.2f} bytes/token")
        print(f"    ├─ UNK rate:            {unk_str:>10}")
        print(f"    └─ Token efficiency:    {(compression_ratio/8*100):.1f}% of byte encoding")
    except Exception as e:
        print(f"  {name:<23} ERROR - {e}")

# ============================================
# 4. TOKEN FREQUENCY DISTRIBUTION
# ============================================
print("\n" + "="*60)
print("4. TOP TOKENS BY FREQUENCY")
print("="*60)

for name, tok_info in tokenizers.items():
    if name not in stats_results:
        continue
    
    try:
        decode_fn = tok_info["decode_single"]
        all_tokens = stats_results[name]['all_tokens']
        freq = Counter(all_tokens)
        
        print(f"\n  {name}:")
        print(f"  {'Token':<25} {'Count':>8}  {'Frequency':>10}")
        print(f"  {'-'*25} {'-'*8}  {'-'*10}")
        total = len(all_tokens)
        for token_id, count in freq.most_common(10):
            try:
                surface = decode_fn(token_id)
                # Clean up display for special chars
                display = surface.replace('\n', '\\n').replace('\t', '\\t')
                if len(display) > 25:
                    display = display[:22] + '...'
                percentage = (count / total * 100)
                print(f"  {display:<25} {count:>8}  {percentage:>9.2f}%")
            except:
                print(f"  [ID:{token_id:>6}]{'':<17} {count:>8}")
    except Exception as e:
        print(f"  {name}: Error - {e}")

# ============================================
# 5. DEVANAGARI COVERAGE ANALYSIS
# ============================================
print("\n" + "="*60)
print("5. DEVANAGARI (नेपाली) CHARACTER COVERAGE")
print("="*60)

# Devanagari Unicode range
devanagari_range = set()
for code in range(0x0900, 0x097F + 1):
    devanagari_range.add(chr(code))

print(f"\n  Total Devanagari Unicode characters: {len(devanagari_range)}")

coverage_results = {}
for name, tok_info in tokenizers.items():
    try:
        vocab_size = tok_info["vocab_size"]()
        decode_fn = tok_info["decode_single"]
        
        devanagari_found = set()
        # Sample from vocabulary (checking all tokens might be slow)
        for token_id in range(min(vocab_size, 100000)):  # Limit to 100k for speed
            try:
                surface = decode_fn(token_id)
                for char in surface:
                    if char in devanagari_range:
                        devanagari_found.add(char)
            except:
                pass
        
        coverage = (len(devanagari_found) / len(devanagari_range)) * 100
        coverage_results[name] = coverage
        
        # Visual bar
        bar_length = int(coverage / 5)  # 20 segments max
        bar = '█' * bar_length + '░' * (20 - bar_length)
        
        print(f"  {name:<23} {bar} {coverage:>5.1f}% ({len(devanagari_found)}/128)")
    except Exception as e:
        print(f"  {name:<23} Could not check coverage")

# ============================================
# 6. TOKENIZATION QUALITY - SAMPLE COMPARISONS
# ============================================
print("\n" + "="*60)
print("6. SAMPLE TOKENIZATION QUALITY")
print("="*60)

sample_texts = [
    "नेपाल एक सुन्दर देश हो",
    "म घर जाँदै छु",
    "हामीले विद्यालयमा पढ्यौं",
    "संविधान सभाबाट पारित भयो",
    "मेरो नाम राम हो र म काठमाडौंमा बस्छु",
]

for text in sample_texts:
    print(f"\n  Input: {text}")
    print(f"  {'─'*65}")
    
    # First show HimalayanTOK prominently
    if HT_NAME in tokenizers:
        try:
            tok_info = tokenizers[HT_NAME]
            tokens = tok_info["encode"](text)
            surfaces = [tok_info["decode_single"](t) for t in tokens]
            surface_str = " | ".join(surfaces)
            print(f"  🏔️  {HT_NAME}: [{surface_str}]")
        except Exception as e:
            print(f"  🏔️  {HT_NAME}: ERROR - {e}")
    
    # Then show others
    for name, tok_info in tokenizers.items():
        if name == HT_NAME:
            continue  # Already shown
        try:
            encode_fn = tok_info["encode"]
            decode_fn = tok_info["decode_single"]
            
            tokens = encode_fn(text)
            surfaces = [decode_fn(t) for t in tokens]
            
            # Format nicely
            surface_str = " | ".join(surfaces)
            print(f"  {name:<23}: [{surface_str}]")
        except Exception as e:
            print(f"  {name:<23}: ERROR - {e}")

# ============================================
# 7. NEPALI MORPHOLOGICAL CONSISTENCY
# ============================================
print("\n" + "="*60)
print("7. NEPALI MORPHOLOGICAL CONSISTENCY")
print("="*60)

# Test morphological consistency
morph_pairs = [
    ("जान्छु", "जान्छौ", "go (I/you)"),
    ("खान्छु", "खान्छौ", "eat (I/you)"),
    ("गर्छु", "गर्छौ", "do (I/you)"),
    ("पढ्छु", "पढ्छौ", "read (I/you)"),
    ("आउँछु", "आउँछौ", "come (I/you)"),
]

print(f"\n  Shared tokens between verb conjugations (higher = better morphology recognition):")
print(f"  {'Pair':<25} {'Meaning':<15}", end="")
for name in tokenizers:
    print(f" {name:<15}", end="")
print()
print(f"  {'─'*25} {'─'*15}{'─'*15 * len(tokenizers)}")

# We'll also collect a morphology score for the final table
morph_scores = {}
for name in tokenizers:
    morph_scores[name] = []

for word1, word2, meaning in morph_pairs:
    print(f"  {word1}/{word2:<18} {meaning:<15}", end="")
    
    for name, tok_info in tokenizers.items():
        try:
            encode_fn = tok_info["encode"]
            
            tokens1 = encode_fn(word1)
            tokens2 = encode_fn(word2)
            
            # Count shared token IDs
            shared = len(set(tokens1) & set(tokens2))
            total1, total2 = len(tokens1), len(tokens2)
            # Fraction of shared tokens relative to max length
            share_frac = shared / max(1, max(total1, total2))
            morph_scores[name].append(share_frac)
            print(f" {shared}/{total1},{total2:<11}", end="")
        except:
            print(f" {'N/A':<13}", end="")
    print()

# Calculate average morphology share
avg_morph = {name: sum(vals)/len(vals) if vals else 0.0 for name, vals in morph_scores.items()}

# ============================================
# 8. FINAL SUMMARY TABLE (without misleading score)
# ============================================
print("\n" + "="*80)
print("8. COMPREHENSIVE BENCHMARK SUMMARY")
print("="*80)

print(f"\n{'Tokenizer':<23} {'Vocab':>8} {'Tokens':>8} {'Tok/Wrd':>8} {'Compr':>6} {'Bytes/Tok':>9} {'UNK%':>7} {'Deva%':>7} {'Speed':>12} {'Morph.Share':>12}")
print(f"{'─'*23} {'─'*8} {'─'*8} {'─'*8} {'─'*6} {'─'*9} {'─'*7} {'─'*7} {'─'*12} {'─'*12}")

for name, tok_info in tokenizers.items():
    try:
        vocab_sz = tok_info["vocab_size"]()
        
        if name in stats_results and name in speed_results and name in coverage_results:
            st = stats_results[name]
            sp = speed_results[name]
            cv = coverage_results[name]
            ms = avg_morph.get(name, 0.0)
            
            print(f"{name:<23} {vocab_sz:>8,} {st['total_tokens']:>8,} "
                  f"{st['avg_tokens_per_word']:>8.3f} {st['compression_ratio']:>5.2f} "
                  f"{st['bytes_per_token']:>8.2f} "
                  f"{st['unk_rate']:>6.2f}% {cv:>6.1f}% "
                  f"{sp['lines_per_sec']:>10,.0f}/s {ms:>11.3f}")
        else:
            print(f"{name:<23} {vocab_sz:>8,} {'N/A':>8} {'N/A':>8} {'N/A':>6} {'N/A':>9} {'N/A':>7} {'N/A':>7} {'N/A':>12} {'N/A':>12}")
    except Exception as e:
        print(f"{name:<23} {'ERROR':>8}")

# Interpretation note
print(f"\n  ⚠️  IMPORTANT NOTES:")
print(f"  - 'Morph.Share' is the average fraction of shared tokens between paired verb forms.")
print(f"    It measures how much the tokenizer recognizes morphological stems; higher is better.")
print(f"  - Tokenizers were used off‑the‑shelf; vocabulary sizes are NOT matched.")
print(f"    Fertility (Tok/Wrd) naturally improves with larger vocabularies. A fair comparison")
print(f"    requires retraining all tokenizers with the same vocabulary budget on the same corpus.")
print(f"  - Speed is influenced by implementation language (C++ vs Python) and should not be directly")
print(f"    interpreted as a purely algorithmic advantage.")
print(f"  - A definitive quality evaluation requires measuring bits‑per‑character (BPC) using")
print(f"    a language model trained on each tokenizer. This benchmark does not include BPC.")

# ============================================
# 9. KEY FINDINGS (honest and balanced)
# ============================================
print("\n" + "="*60)
print("9. KEY FINDINGS & CAVEATS")
print("="*60)

if HT_NAME in stats_results and HT_NAME in speed_results:
    ht_speed = speed_results[HT_NAME]['lines_per_sec']
    ht_stats = stats_results[HT_NAME]
    ht_coverage = coverage_results.get(HT_NAME, 0)
    ht_morph = avg_morph.get(HT_NAME, 0)
    
    print(f"\n  🏔️  {HT_NAME} Highlights:")
    print(f"     ├─ Speed: {ht_speed:,.0f} lines/s (fastest, but C++ implementation)")
    print(f"     ├─ Devanagari Coverage: {ht_coverage:.1f}% (best)")
    print(f"     ├─ Tokens/word: {ht_stats['avg_tokens_per_word']:.3f} (competitive)")
    print(f"     └─ UNK rate: 0.00% (byte‑fallback)")
    print(f"\n  ⚠️  Limitations:")
    print(f"     ├─ Morphological token sharing: {ht_morph:.3f} (worst among tokenizers tested)")
    print(f"     │   This indicates the tokenizer does NOT currently benefit from explicit morphology.")
    print(f"     ├─ Vocabulary size is {ht_vocab_size:,}, not the marketed '64K'.")
    print(f"     └─ Comparisons above are against off‑the‑shelf tokenizers; vocab budgets are not matched.")
    print(f"\n  💡 Honest Assessment:")
    print(f"     {HT_NAME} is a fast, byte‑fallback BPE tokenizer with perfect Devanagari coverage.")
    print(f"     Its raw compression (tokens/word) is strong for its vocabulary size, but the")
    print(f"     morphological design appears inactive (zero shared stem tokens).")
    print(f"     For a publishable claim, you must:")
    print(f"       1) Retrain baselines at the same vocab budget on the same corpus.")
    print(f"       2) Measure BPC on a language model.")
    print(f"       3) Either fix the FST‑to‑tokenizer pipeline or reframe as a frequency‑BPE resource.")
    print(f"     As it stands, this is a useful resource tokenizer, not a morphological breakthrough.")
else:
    print(f"\n  Unable to display findings for {HT_NAME} (missing data).")

print("\n" + "="*60)
print("BENCHMARK COMPLETE")
print("="*60)

try:
    benchmark_results = {
        'tokenizer_name': HT_NAME,
        'actual_vocab_size': ht_vocab_size,
        'corpus': DATASET,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'speed_results': {k: {'lines_per_sec': v['lines_per_sec']} for k, v in speed_results.items()},
        'stats_results': {k: {
            'total_tokens': v['total_tokens'],
            'avg_tokens_per_word': v['avg_tokens_per_word'],
            'compression_ratio': v['compression_ratio'],
            'unk_rate': v['unk_rate'],
        } for k, v in stats_results.items()},
        'coverage_results': coverage_results,
        'morph_share': avg_morph,
    }
    
    with open('himalayantok_benchmark_results.json', 'w', encoding='utf-8') as f:
        json.dump(benchmark_results, f, indent=2, ensure_ascii=False)
    print(f"\n📁 Results saved to: himalayantok_benchmark_results.json")
except Exception as e:
    print(f"\n⚠️  Could not save results to JSON: {e}")

LOADING TOKENIZERS
Loading HimalayanTOK-Nepali...
  ⚠️  WARNING: Loaded vocabulary size is 100,001, not 64,000.
     The tokenizer will be named 'HimalayanTOK-100K' to reflect reality.
Loading IndicBERT...


[load] warning: /home/lang-chain/Documents/HimalayanTokenizer/vocab_nepbpe/nepbpe_vocab_bilingual_v10.tsv has no mode header; assuming mode=LM


Loading mBERT...
Loading XLM-R...
Loading Tiktoken...
Loading Llama tokenizer...

Loading corpus...

BENCHMARK: HimalayanTOK-100K vs Major Tokenizers
Corpus: sample.txt
Total lines: 596
Total words: 5894
Total characters: 40407

1. TOKENIZER INFORMATION & VOCABULARY SIZE

Tokenizer                   Vocab Size Type                
------------------------- ------------ --------------------
  HimalayanTOK-100K            100,001  CUSTOM              
    └─ HimalayanTOK-Nepali (BPE, byte-fallback, 100% Devanagari)
  IndicBERT                    200,000  HF                  
    └─ AI4Bharat IndicBERT (200K vocab, WordPiece)
  mBERT                        119,547  HF                  
    └─ BERT-base-multilingual-cased (119K vocab, WordPiece)
  XLM-R                        250,002  HF                  
    └─ XLM-RoBERTa-base (250K vocab, SentencePiece)
  TikToken-cl100k              100,277  TIKTOKEN            
    └─ OpenAI TikToken cl100k_base (100K vocab, BPE)
  TikToken-o200k     